Env setup and import

In [12]:
import torch
from torch import nn
from torchrl.envs import PettingZooWrapper
from env_simplified import MahjongGameEnv
from torchrl.envs import TransformedEnv
from torchrl.envs.transforms import ActionMask
from torchrl.envs.utils import MarlGroupMapType
from torchrl.modules import MultiAgentConvNet, MultiAgentMLP, ProbabilisticActor, MaskedCategorical
from tensordict.nn import TensorDictModule
from torchrl.collectors import Collector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.objectives import ClipPPOLoss, ValueEstimators

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

env = PettingZooWrapper(
    env=MahjongGameEnv(),
    use_mask=True,
    return_state=True,
    categorical_actions=True,
    group_map=MarlGroupMapType.ALL_IN_ONE_GROUP
)


cuda


c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\torchrl\envs\libs\pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


Flattened Policy

In [13]:
# class CastToFloat(nn.Module):
#     def forward(self, x):
#         return x.float()   # or .to(torch.float32)

# policy_net = nn.Sequential(
#     nn.Flatten(-2),
#     CastToFloat(),
#     MultiAgentMLP(
#         n_agent_inputs = 29 * 34,       
#         n_agent_outputs = 75,       
#         n_agents = 4,
#         centralized = False,        
#         share_params = True,      
#         depth = 8,               
#         num_cells = 1024,
#         activation_class=torch.nn.Tanh
#     )
# )

# # 1. 讀取原始權重
# raw_state_dict = torch.load('actor_net_4.pth')

# # 2. 獲取你目前 policy_net 真正需要的 Keys 列表
# model_keys = list(policy_net.state_dict().keys())

# # 3. 過濾掉檔案中重複的鍵，只按順序保留獨一無二的權重
# #（利用 dict.fromkeys 保持順序並去重，移除多餘的 params 或 _empty_net 命名空間干擾）
# unique_raw_keys = list(dict.fromkeys([k.replace('params', '_empty_net') for k in raw_state_dict.keys()]))
# # 還原回檔案中實際存在的正確鍵名
# src_keys = []
# for k in unique_raw_keys:
#     if k in raw_state_dict:
#         src_keys.append(k)
#     else:
#         src_keys.append(k.replace('_empty_net', 'params'))

# # 4. 建立一對一的對齊字典
# aligned_state_dict = {}
# for m_key, s_key in zip(model_keys, src_keys):
#     aligned_state_dict[m_key] = raw_state_dict[s_key]
#     # 打印對齊狀態供你確認，成功後可刪除此行
#     print(f"對齊成功: {s_key} -> {m_key}")
    
# policy_net.load_state_dict(aligned_state_dict)

# policy_module = TensorDictModule(
#     policy_net,
#     in_keys=[("agents", "observation", "observation")],
#     out_keys=[("agents", "logits")],
# )

# policy = ProbabilisticActor(
#     module=policy_module,
#     spec=env.action_spec_unbatched,
#     in_keys={
#         'logits': ('agents', 'logits'),
#         'mask': ('agents', 'action_mask')
#     }, # type: ignore
#     out_keys=[env.action_key],
#     distribution_class=MaskedCategorical,
#     return_log_prob=True
# )  # we'll need the log-prob for the PPO loss

New policy



In [86]:
class CastToFloat(nn.Module):
    def forward(self, x):
        return x.float()   # or .to(torch.float32)

class Unsqueeze(nn.Module):
    def forward(self, x):
        return x.unsqueeze(-3)

policy_net = nn.Sequential(
    CastToFloat(),
    Unsqueeze(),
    MultiAgentConvNet(
        n_agents=4,
        centralized=False,
        share_params=True,
        num_cells=[32, 32, 32],
        paddings=1,
        strides=1,
        kernel_sizes=3,
    ),
    MultiAgentMLP(
        n_agents=4,
        n_agent_inputs=None,
        n_agent_outputs=75,
        num_cells=256,
        centralized=False,
        share_params=True,
        depth=2,
    )
)

policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation", "observation")],
    out_keys=[("agents", "logits")],
)

policy = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec_unbatched,
    in_keys={
        'logits': ('agents', 'logits'),
        'mask': ('agents', 'action_mask')
    }, # type: ignore
    out_keys=[env.action_key],
    distribution_class=MaskedCategorical,
    return_log_prob=True
)  # we'll need the log-prob for the PPO loss

Critic

In [15]:
# critic_net = nn.Sequential(
#     nn.Flatten(-2),                    # [248,46] -> [248*46]
#     CastToFloat(),
#     nn.Linear(52 * 42, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),                # output 4 values (one per agent)
#     nn.ReLU(),
#     nn.Linear(512, 512),                # output 4 values (one per agent)
#     nn.ReLU(),
#     nn.Linear(512, 4),                # output 4 values (one per agent)
#     nn.Unflatten(-1, (4, 1))          # reshape from [4] to [4,1]
# )
# # 1. 載入原始的 state_dict
# state_dict = torch.load('critic_net_new_1.pth')

# # 2. 移除所有鍵值（Keys）開頭的 "module." 前綴
# from collections import OrderedDict
# new_state_dict = OrderedDict()
# for k, v in state_dict.items():
#     name = k.replace("module.", "") # 如果開頭有 module. 就拿掉
#     new_state_dict[name] = v

# # 3. 將乾淨的 state_dict 載入到你的 critic_net
# critic_net.load_state_dict(new_state_dict)

# critic = TensorDictModule(
#     module=critic_net,
#     in_keys=["state"],               # global state
#     out_keys=[("agents", "state_value")],  # shape [4] values under agents
# )

In [93]:
class UnsqueezeV2(nn.Module):
    def forward(self, x):
        return x.unsqueeze(-3)
critic_net = nn.Sequential(
    CastToFloat(),
    UnsqueezeV2(),
    nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
    # nn.BatchNorm2d(32),
    nn.ReLU(),
    # nn.Dropout2d(0.5),  # Dropout2d is recommended for conv layers (channel-wise)
    
    # Block 2
    nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1),
    # nn.BatchNorm2d(64),
    nn.ReLU(),
    # nn.Dropout2d(0.5),
    
    # Block 3
    nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1),
    # nn.BatchNorm2d(128),
    nn.ReLU(),
    # nn.Dropout2d(0.5),
    
    # MLP Head (standard MultiAgentMLP style after flatten)
    nn.Flatten(-3),
    nn.Linear(69888, 4),  
    nn.Unflatten(-1, (4, 1))
)
critic = TensorDictModule(
    module=critic_net,
    in_keys=["state"],               # global state
    out_keys=[("agents", "state_value")],  # shape [4] values under agents
)

Initialize the model

In [94]:
policy = policy.to('cpu')
critic = critic.to('cpu')
print("Running policy:", policy(env.reset()))
print("Running value:", critic(env.reset()))

Running policy: TensorDict(
    fields={
        agents: TensorDict(
            fields={
                action: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.int64, is_shared=False),
                action_log_prob: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.float32, is_shared=False),
                action_mask: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.bool, is_shared=False),
                done: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                logits: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.float32, is_shared=False),
                mask: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: TensorDict(
                    fields={
                        observation: Tensor(shape=torch.Size([4, 29, 34]), device=cpu, dtype=torch.uint8, is_shared=False)},
                    batch_size=torch.Size([4]),
                    device=

Load models

In [96]:
policy_net.load_state_dict(torch.load('actor_net (9).pth'))
critic.load_state_dict(torch.load("critic_net (5).pth"))

<All keys matched successfully>

Rollout

In [97]:
import pygame
from pygame_visualizer import render_game_state

pygame.init()
data = env.rollout(200, policy=policy.to('cpu'))
print(data)
screen = pygame.display.set_mode(size=(800, 800))
font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)
from sys import exit
while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            exit()
    screen.fill('white')
    render_game_state(env._env.gamestate, screen, font)
    pygame.display.update()


TensorDict(
    fields={
        agents: TensorDict(
            fields={
                action: Tensor(shape=torch.Size([106, 4]), device=cpu, dtype=torch.int64, is_shared=False),
                action_log_prob: Tensor(shape=torch.Size([106, 4]), device=cpu, dtype=torch.float32, is_shared=False),
                action_mask: Tensor(shape=torch.Size([106, 4, 75]), device=cpu, dtype=torch.bool, is_shared=False),
                done: Tensor(shape=torch.Size([106, 4, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                logits: Tensor(shape=torch.Size([106, 4, 75]), device=cpu, dtype=torch.float32, is_shared=False),
                mask: Tensor(shape=torch.Size([106, 4]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: TensorDict(
                    fields={
                        observation: Tensor(shape=torch.Size([106, 4, 29, 34]), device=cpu, dtype=torch.uint8, is_shared=False)},
                    batch_size=torch.Size([106, 4]),
   

SystemExit: 

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Loss function and optimizer

In [108]:
loss_module = ClipPPOLoss(
    actor_network=policy,
    critic_network=critic,
    entropy_coeff=0.02
)
loss_module.set_keys(  # We have to tell the loss where to find the keys
    reward=env.reward_key,
    action=env.action_key,
    value=("agents", "state_value"),
    # These last 2 keys will be expanded to match the reward shape
    done=("agents", "done"),                # per-agent
    terminated=("agents", "terminated"),
)
gamma = 0.995  # discount factor
lmbda = 0.9  # lambda for generalised advantage estimation
lr = 5e-7
loss_module.make_value_estimator(
    ValueEstimators.GAE, gamma=gamma, lmbda=lmbda
)  
GAE = loss_module.value_estimator

optim = torch.optim.Adam(loss_module.parameters(), lr)

loss_module = loss_module.to(device)

In [107]:
policy_net.state_dict()

OrderedDict([('2.params.0.weight',
              tensor([[[[ 2.4765e-01,  2.2581e-01, -1.9558e-01],
                        [ 9.1834e-03, -9.9854e-02, -2.0666e-01],
                        [ 4.4338e-03,  1.1367e-01,  2.3882e-02]]],
              
              
                      [[[ 2.6856e-01,  1.3536e-01, -2.6382e-01],
                        [-1.2093e-01, -1.8504e-01, -4.6304e-02],
                        [ 2.7579e-01,  3.8275e-02, -4.9681e-02]]],
              
              
                      [[[-3.7998e-02,  6.7279e-02, -2.8518e-01],
                        [ 4.9497e-02, -1.9879e-01,  1.9179e-01],
                        [-8.9740e-03, -1.1232e-01, -2.1395e-01]]],
              
              
                      [[[-3.3043e-01,  6.3818e-02, -8.8916e-02],
                        [ 3.2201e-01,  2.9730e-01, -3.0756e-01],
                        [ 2.5837e-01,  3.2265e-01, -3.0301e-01]]],
              
              
                      [[[ 2.1756e-01,  3.2963e-01,  1.064

Train loop

In [100]:
num_epochs = 5
max_grad_norm = 0.1
frames_per_batch = 1000  # Number of team frames collected per training iteration
n_iters = 200  # Number of sampling and training iterations
total_frames = frames_per_batch * n_iters
minibatch_size = 1000

replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(
        frames_per_batch, device=device
    ),  # We store the frames_per_batch collected at each iteration
    sampler=SamplerWithoutReplacement(),
    batch_size=minibatch_size,  # We will sample minibatches of this siz
)
collector= Collector(
    env,
    policy,
    device='cpu',
    storing_device=device,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames
)

from tqdm.auto import tqdm
for tensordict_data in tqdm(collector):
    tensordict_data.set(
        ("next", "agents", "done"),
        tensordict_data.get(("next", "done"))
        .unsqueeze(-1)
        .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
    )
    tensordict_data.set(
        ("next", "agents", "terminated"),
        tensordict_data.get(("next", "terminated"))
        .unsqueeze(-1)
        .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
    )
    # We need to expand the done and terminated to match the reward shape (this is expected by the value estimator)

    with torch.no_grad():
        GAE(
            tensordict_data,
            params=loss_module.critic_network_params,
            target_params=loss_module.target_critic_network_params,
        )  # Compute GAE and add it to the data

    data_view = tensordict_data.reshape(-1)  # Flatten the batch size to shuffle data
    replay_buffer.extend(data_view)

    for _ in range(num_epochs):
        for _ in range(frames_per_batch // minibatch_size):
            subdata = replay_buffer.sample()
            subdata = subdata.to(device)
            loss_vals = loss_module(subdata)

            loss_value = (
                loss_vals["loss_objective"]
                + loss_vals["loss_critic"]
                + loss_vals["loss_entropy"]
            )

            loss_value.backward()

            torch.nn.utils.clip_grad_norm_(
                loss_module.parameters(), max_grad_norm
            )  # Optional

            optim.step()
            optim.zero_grad()

    collector.update_policy_weights_()

# After training
torch.save(policy_net.state_dict(), "actor_net.pth")
torch.save(critic.state_dict(), "critic_net.pth")
# files.download('actor_net.pth')
# files.download('critic_net.pth')

C:\Users\ctc73\AppData\Local\Temp\ipykernel_34748\2633037421.py:15: FutureWarning: The env passed to Collector is missing transforms required by the policy (InitTracker). From torchrl v0.15 the collector will append them automatically. To enable that behavior now (and silence this warning), pass `auto_register_policy_transforms=True`. To opt out permanently, pass `auto_register_policy_transforms=False`.
  collector= Collector(
c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\tensordict\_td.py:612: FutureWarning: TensorDict.to_module() is replacing an existing nn.Parameter in the destination module with a tensor leaf that is not an nn.Parameter. This historical behavior can remove the key from module.state_dict(). In tensordict v0.14, to_module() will preserve existing module parameter and buffer registrations by default. Pass preserve_module_state=False to keep the current replacement behavior, or preserve_module_state=True to opt in to the v0.14 behavior now.
  local_o

2026-07-22 19:08:23,411 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([1000]) shape [END]


  6%|▌         | 12/200 [08:00<2:03:36, 39.45s/it]

Yaku counted (with values):
  flowers (combined): 2
  tsumo: 1
  ping_hu: 1


 18%|█▊        | 37/200 [24:34<1:48:16, 39.86s/it]

...

 93%|█████████▎| 186/200 [2:10:04<09:01, 38.70s/it]

hua_hu: 3


 94%|█████████▍| 188/200 [2:11:24<07:50, 39.24s/it]

Yaku counted (with values):
  flowers (combined): 2
  tsumo: 1


100%|██████████| 200/200 [2:19:17<00:00, 41.79s/it]


In [105]:
policy_net.state_dict()
another_state_dict = torch.load("actor_net (9).pth")
print(another_state_dict['2.params.0.bias'])
print(policy_net.state_dict()['2.params.0.bias'])

tensor([ 0.1325,  0.1813,  0.0202,  0.0732,  0.2762, -0.0645, -0.0455,  0.3278,
        -0.2483, -0.1929,  0.3147,  0.1084, -0.1188, -0.0912, -0.1158,  0.2326,
        -0.1551, -0.1653,  0.1431, -0.2901,  0.2374,  0.3277,  0.0840, -0.2914,
        -0.1911,  0.3236, -0.0545,  0.1149,  0.2313,  0.1552, -0.0719,  0.2461],
       device='cuda:0')
tensor([ 0.1323,  0.1814,  0.0200,  0.0730,  0.2759, -0.0643, -0.0458,  0.3277,
        -0.2484, -0.1928,  0.3146,  0.1082, -0.1186, -0.0910, -0.1160,  0.2324,
        -0.1552, -0.1654,  0.1429, -0.2903,  0.2374,  0.3280,  0.0838, -0.2915,
        -0.1909,  0.3235, -0.0543,  0.1150,  0.2312,  0.1550, -0.0718,  0.2460],
       device='cuda:0')


In [70]:
trainable_params = sum(p.numel() for p in critic_net.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

Trainable parameters: 1,210,884
